## ディレクトリの削除

### import

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import paramiko
from scp import SCPClient
import os
import pandas as pd
import numpy as np
import re
import time
import subprocess
import ast
from datetime import datetime
import shutil
from datetime import datetime

from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split,KFold, cross_validate,cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error,make_scorer
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.metrics import log_loss
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from itertools import product
import gc
import random
from torch.utils.data import Dataset, DataLoader
import copy
import math
import posixpath
import shlex
import json
import base64

### サーバー接続

In [4]:
def connect_server():
    try:
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        # あなたのサーバー情報に合わせて書き換えてください
        client.connect('192.168.11.20', username='teraimao', key_filename='/Users/teraimao/.ssh/id_ed25519')
        return client
    except Exception as e:
        print(f"Connection failed: {e}")
        return None

# 関数を呼び出して、戻り値を 'client' という変数に入れる
client = connect_server()

if client:
    print("✅ サーバーへの接続に成功しました。'client' が定義されました。")
else:
    print("❌ 接続に失敗しました。設定を確認してください。")

✅ サーバーへの接続に成功しました。'client' が定義されました。


In [8]:
# ============================================================
# ID範囲指定 HDGBvdW workディレクトリ削除UI
# サーバー側・Mac側をプレビュー確認後にまとめて削除する
# ============================================================

# ============================================================
# 安全設定
# ============================================================

# この範囲外のディレクトリは削除できないようにする
SAFE_LOCAL_ROOT = "/Users/teraimao/experiment"
SAFE_SERVER_ROOT = "/home/teraimao/experiment"

# 今回はHDGBvdW用。誤ってHDGB/GBSWを消さないための追加安全条件
REQUIRED_NAME_TOKEN = "_hdgbvdw_work"


# ============================================================
# GUI
# ============================================================

common_style = {"description_width": "150px"}
wide_layout = widgets.Layout(width="900px")

ssh_target_widget = widgets.Text(
    value="frigate",
    description="SSH接続先:",
    style=common_style,
    layout=widgets.Layout(width="500px"),
)

server_base_widget = widgets.Text(
    value="/home/teraimao/experiment/confirm",
    description="Server work base:",
    style=common_style,
    layout=wide_layout,
)

local_base_widget = widgets.Text(
    value="/Users/teraimao/experiment/confirm",
    description="Mac work base:",
    style=common_style,
    layout=wide_layout,
)

start_id_widget = widgets.BoundedIntText(
    value=1334,
    min=0,
    max=99999999,
    description="開始ID:",
    style=common_style,
    layout=widgets.Layout(width="350px"),
)

end_id_widget = widgets.BoundedIntText(
    value=1600,
    min=0,
    max=99999999,
    description="終了ID:",
    style=common_style,
    layout=widgets.Layout(width="350px"),
)

name_filter_widget = widgets.Text(
    value="2026.07.21_500ps_hdgbvdw_work",
    description="名前に含む文字:",
    style=common_style,
    layout=wide_layout,
)

delete_server_widget = widgets.Checkbox(
    value=True,
    description="サーバー側を削除",
    indent=False,
)

delete_local_widget = widgets.Checkbox(
    value=True,
    description="Mac側も削除",
    indent=False,
)

preview_button = widgets.Button(
    description="🔍 削除候補を確認",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="220px", height="42px"),
)

confirm_widget = widgets.Checkbox(
    value=False,
    description="表示された候補を削除してよい",
    indent=False,
)

delete_button = widgets.Button(
    description="🗑 対象ディレクトリを削除",
    button_style="danger",
    icon="trash",
    disabled=True,
    layout=widgets.Layout(width="250px", height="42px"),
)

output_widget = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #cccccc",
        padding="10px",
        width="1000px",
        max_height="450px",
        overflow="auto",
    )
)

preview_state = {}


# ============================================================
# 共通処理
# ============================================================

def validate_settings():
    start_id = start_id_widget.value
    end_id = end_id_widget.value
    token = name_filter_widget.value.strip()

    if start_id > end_id:
        raise ValueError("開始IDは終了ID以下にしてください。")

    if end_id - start_id > 5000:
        raise ValueError("一度に削除できる範囲は5000 IDまでです。")

    # 空欄や短すぎる文字列による広範囲削除を防ぐ
    if len(token) < 4:
        raise ValueError(
            "「名前に含む文字」は4文字以上入力してください。\n"
            "例: 2026.07.21_500ps_hdgbvdw_work"
        )

    # HDGB/GBSW誤削除防止
    if REQUIRED_NAME_TOKEN not in token:
        raise ValueError(
            f"安全のため、名前フィルタには必ず {REQUIRED_NAME_TOKEN} を含めてください。\n"
            "例: 2026.07.21_500ps_hdgbvdw_work"
        )

    if not delete_server_widget.value and not delete_local_widget.value:
        raise ValueError(
            "「サーバー側を削除」または「Mac側も削除」を選択してください。"
        )


def settings_signature():
    return (
        ssh_target_widget.value.strip(),
        server_base_widget.value.strip(),
        local_base_widget.value.strip(),
        start_id_widget.value,
        end_id_widget.value,
        name_filter_widget.value.strip(),
        delete_server_widget.value,
        delete_local_widget.value,
    )


def is_safe_path(path, safe_root):
    path = os.path.realpath(os.path.expanduser(path))
    safe_root = os.path.realpath(os.path.expanduser(safe_root))

    try:
        inside_root = os.path.commonpath([path, safe_root]) == safe_root
    except ValueError:
        return False

    # experiment直下そのものは削除対象の基点にできない
    return inside_root and path != safe_root


def scan_local_directories():
    base = os.path.realpath(
        os.path.expanduser(local_base_widget.value.strip())
    )
    start_id = start_id_widget.value
    end_id = end_id_widget.value
    token = name_filter_widget.value.strip()

    if not is_safe_path(base, SAFE_LOCAL_ROOT):
        raise ValueError(
            f"Mac側のパスが安全範囲外です。\n"
            f"許可範囲: {SAFE_LOCAL_ROOT} 以下"
        )

    if not os.path.isdir(base):
        raise FileNotFoundError(f"Mac側ディレクトリがありません:\n{base}")

    names = []

    for entry in os.scandir(base):
        # シンボリックリンクは対象外
        if not entry.is_dir(follow_symlinks=False):
            continue

        match = re.match(r"^(\d+)_", entry.name)
        if not match:
            continue

        directory_id = int(match.group(1))

        if start_id <= directory_id <= end_id and token in entry.name:
            names.append(entry.name)

    names.sort(key=lambda name: (int(name.split("_", 1)[0]), name))

    return {
        "base": base,
        "names": names,
    }


# ============================================================
# サーバー側で実行するPythonコード
# ============================================================

REMOTE_SCRIPT = r'''
import os
import re
import sys
import json
import base64
import shutil

SAFE_ROOT = os.path.realpath("/home/teraimao/experiment")
REQUIRED_NAME_TOKEN = "_hdgbvdw_work"

base = os.path.realpath(os.path.expanduser(sys.argv[1]))
start_id = int(sys.argv[2])
end_id = int(sys.argv[3])
token = sys.argv[4]
action = sys.argv[5]
expected_encoded = sys.argv[6]


def fail(message):
    print(json.dumps(
        {"ok": False, "error": message},
        ensure_ascii=False
    ))
    sys.exit(1)


try:
    if os.path.commonpath([base, SAFE_ROOT]) != SAFE_ROOT:
        fail("サーバー側のパスが安全範囲外です: " + base)

    if base == SAFE_ROOT:
        fail(
            "安全のため /home/teraimao/experiment "
            "直下は指定できません。"
        )

    if not os.path.isdir(base):
        fail("サーバー側ディレクトリがありません: " + base)

    if REQUIRED_NAME_TOKEN not in token:
        fail(
            "安全のため、名前フィルタには必ず "
            + REQUIRED_NAME_TOKEN
            + " を含めてください。"
        )

    names = []

    for entry in os.scandir(base):
        # 直下の通常ディレクトリだけを対象にする
        if not entry.is_dir(follow_symlinks=False):
            continue

        match = re.match(r"^(\d+)_", entry.name)
        if not match:
            continue

        directory_id = int(match.group(1))

        if (
            start_id <= directory_id <= end_id
            and token in entry.name
        ):
            names.append(entry.name)

    names.sort(
        key=lambda name: (
            int(name.split("_", 1)[0]),
            name
        )
    )

    if action == "preview":
        print(json.dumps(
            {
                "ok": True,
                "base": base,
                "names": names,
                "deleted": 0,
            },
            ensure_ascii=False
        ))
        sys.exit(0)

    if action != "delete":
        fail("不明な操作です: " + action)

    expected_names = json.loads(
        base64.b64decode(expected_encoded).decode("utf-8")
    )

    # プレビュー時から候補が変化していたら削除しない
    if names != expected_names:
        fail(
            "プレビュー後にディレクトリ一覧が変化しました。"
            "もう一度候補確認を実行してください。"
        )

    deleted = []

    for name in names:
        target = os.path.join(base, name)

        # 念のため再度確認
        if not os.path.isdir(target):
            fail("削除対象がディレクトリではありません: " + target)

        if os.path.islink(target):
            fail("シンボリックリンクは削除しません: " + target)

        if REQUIRED_NAME_TOKEN not in name:
            fail("HDGBvdW work以外は削除しません: " + target)

        shutil.rmtree(target)
        deleted.append(name)

    print(json.dumps(
        {
            "ok": True,
            "base": base,
            "names": deleted,
            "deleted": len(deleted),
        },
        ensure_ascii=False
    ))

except Exception as exc:
    fail(str(exc))
'''


def run_remote_action(action, expected_names=None):
    target = ssh_target_widget.value.strip()

    if not target:
        raise ValueError("SSH接続先を入力してください。")

    if expected_names is None:
        expected_names = []

    encoded_names = base64.b64encode(
        json.dumps(expected_names).encode("utf-8")
    ).decode("ascii")

    command = [
        "ssh",
        "-o",
        "BatchMode=yes",
        target,
        "python3",
        "-",
        server_base_widget.value.strip(),
        str(start_id_widget.value),
        str(end_id_widget.value),
        name_filter_widget.value.strip(),
        action,
        encoded_names,
    ]

    result = subprocess.run(
        command,
        input=REMOTE_SCRIPT,
        text=True,
        capture_output=True,
        timeout=180,
    )

    stdout_lines = [
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip()
    ]

    parsed = None

    if stdout_lines:
        try:
            parsed = json.loads(stdout_lines[-1])
        except json.JSONDecodeError:
            parsed = None

    if result.returncode != 0:
        if parsed and parsed.get("error"):
            raise RuntimeError(parsed["error"])

        raise RuntimeError(
            "SSHまたはサーバー処理でエラーが発生しました。\n\n"
            f"標準出力:\n{result.stdout}\n\n"
            f"標準エラー:\n{result.stderr}"
        )

    if not parsed:
        raise RuntimeError(
            "サーバーから結果を読み取れませんでした。\n"
            f"{result.stdout}\n{result.stderr}"
        )

    if not parsed.get("ok"):
        raise RuntimeError(parsed.get("error", "不明なエラー"))

    return parsed


def display_candidates(label, result):
    names = result["names"]

    print(f"\n【{label}】")
    print(f"基点: {result['base']}")
    print(f"候補数: {len(names)} 件")

    if not names:
        print("該当するディレクトリはありません。")
        return

    if len(names) <= 40:
        for name in names:
            print(f"  {name}")
    else:
        for name in names[:20]:
            print(f"  {name}")

        print(f"  ……途中 {len(names) - 40} 件省略……")

        for name in names[-20:]:
            print(f"  {name}")


# ============================================================
# ボタン処理
# ============================================================

def preview_directories(_):
    with output_widget:
        output_widget.clear_output()

        try:
            validate_settings()

            preview_state.clear()

            print("=" * 70)
            print("削除候補を検索しています")
            print("=" * 70)
            print(
                f"ID範囲: {start_id_widget.value}"
                f" ～ {end_id_widget.value}"
            )
            print(
                "名前に含む文字: "
                f"{name_filter_widget.value.strip()}"
            )

            if delete_server_widget.value:
                server_result = run_remote_action("preview")
                preview_state["server"] = server_result
                display_candidates("サーバー側", server_result)

            if delete_local_widget.value:
                local_result = scan_local_directories()
                preview_state["local"] = local_result
                display_candidates("Mac側", local_result)

            preview_state["signature"] = settings_signature()

            total = sum(
                len(preview_state[key]["names"])
                for key in ("server", "local")
                if key in preview_state
            )

            print("\n" + "=" * 70)
            print(f"合計削除候補: {total} 件")

            if total == 0:
                print("削除対象はありません。")
                delete_button.disabled = True
            else:
                print(
                    "内容を確認し、"
                    "「表示された候補を削除してよい」に"
                    "チェックしてください。"
                )
                delete_button.disabled = False

            confirm_widget.value = False

        except Exception as exc:
            preview_state.clear()
            delete_button.disabled = True
            confirm_widget.value = False

            print("エラー:")
            print(exc)


def delete_directories(_):
    with output_widget:
        try:
            if not preview_state:
                raise RuntimeError(
                    "先に「削除候補を確認」を押してください。"
                )

            if settings_signature() != preview_state.get("signature"):
                raise RuntimeError(
                    "プレビュー後に設定が変更されています。\n"
                    "もう一度「削除候補を確認」を押してください。"
                )

            if not confirm_widget.value:
                raise RuntimeError(
                    "確認チェックを入れてから削除してください。"
                )

            print("\n" + "=" * 70)
            print("削除を開始します")
            print("=" * 70)

            # サーバー側
            if "server" in preview_state:
                expected = preview_state["server"]["names"]

                result = run_remote_action(
                    "delete",
                    expected_names=expected,
                )

                print(
                    f"サーバー側: "
                    f"{result['deleted']} 件削除しました。"
                )

            # Mac側
            if "local" in preview_state:
                expected = preview_state["local"]["names"]

                # 削除直前に候補が変わっていないか確認
                current = scan_local_directories()

                if current["names"] != expected:
                    raise RuntimeError(
                        "Mac側のディレクトリ一覧が"
                        "プレビュー時から変化しました。\n"
                        "もう一度候補確認を実行してください。"
                    )

                for name in expected:
                    target = os.path.join(
                        current["base"],
                        name,
                    )

                    if os.path.islink(target):
                        raise RuntimeError(
                            "シンボリックリンクは削除しません:\n"
                            + target
                        )

                    if REQUIRED_NAME_TOKEN not in name:
                        raise RuntimeError(
                            "HDGBvdW work以外は削除しません:\n"
                            + target
                        )

                    shutil.rmtree(target)

                print(
                    f"Mac側: {len(expected)} 件削除しました。"
                )

            print("\n削除が完了しました。")

            preview_state.clear()
            delete_button.disabled = True
            confirm_widget.value = False

        except Exception as exc:
            print("\n削除処理を中止しました:")
            print(exc)


def invalidate_preview(change=None):
    preview_state.clear()
    delete_button.disabled = True
    confirm_widget.value = False


preview_button.on_click(preview_directories)
delete_button.on_click(delete_directories)

for widget in [
    ssh_target_widget,
    server_base_widget,
    local_base_widget,
    start_id_widget,
    end_id_widget,
    name_filter_widget,
    delete_server_widget,
    delete_local_widget,
]:
    widget.observe(invalidate_preview, names="value")


# ============================================================
# 表示
# ============================================================

ui = widgets.VBox([
    widgets.HTML(
        "<h2>🗑 ID範囲指定 HDGBvdW workディレクトリ削除</h2>"
        "<p>指定したID範囲に一致するHDGBvdW作業ディレクトリを、"
        "確認後にまとめて削除します。</p>"
    ),

    ssh_target_widget,

    widgets.HTML("<h3>1. 削除対象の場所</h3>"),
    server_base_widget,
    local_base_widget,

    widgets.HTML("<h3>2. ID範囲・ディレクトリ名</h3>"),
    widgets.HBox([
        start_id_widget,
        end_id_widget,
    ]),
    name_filter_widget,

    widgets.HTML("<h3>3. 削除場所</h3>"),
    widgets.HBox([
        delete_server_widget,
        delete_local_widget,
    ]),

    widgets.HTML("<h3>4. 実行</h3>"),
    widgets.HBox([
        preview_button,
        confirm_widget,
        delete_button,
    ]),

    output_widget,
])

display(ui)

1333_2026.07.21_500ps_hdgbvdw_work

## 後ろ